# Wetland Training Dataset Creator - OPTIMIZED

**Output:** `wetland_dataset_1.5M_4Training.npz`

**Optimizations:**
- ✅ Vectorized pixel extraction (100x faster)
- ✅ Reduced Google Drive I/O calls
- ✅ Batch processing
- ✅ Memory efficient

In [13]:
# CELL 1: Setup
print("🚀 Setting up environment...")

import os
import sys
from google.colab import drive

# Mount Google Drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')
else:
    print("✓ Drive already mounted")

# Install dependencies
!pip install -q rasterio tqdm

import numpy as np
import torch
import rasterio
from pathlib import Path
from tqdm import tqdm

print("✅ Setup complete!")

🚀 Setting up environment...
✓ Drive already mounted
✅ Setup complete!


In [14]:
# CELL 2: Configuration
print("="*70)
print("CONFIGURATION: CNN SPATIAL DATASET")
print("="*70)

# CNN Hyperparameters - REDUCED PATCH SIZE for deep learning stability
PATCH_SIZE = 15
PATCH_RADIUS = PATCH_SIZE // 2
TRAIN_SPLIT = 0.8
SEED = 42

# Paths
labels_file = "/content/drive/MyDrive/bow_river_wetlands_10m_final.tif"
embeddings_dir = Path("/content/drive/MyDrive/EarthEngine")
output_file = "/content/drive/MyDrive/wetland_cnn_dataset_15x15.npz"

# Substantially increased sampling limits due to smaller patch size
TRAIN_SAMPLES_PER_CLASS = 10_000
VAL_SAMPLES_PER_CLASS = 2_000

print(f"\nLabels: {labels_file}")
print(f"Embeddings: {embeddings_dir}")
print(f"Output: {output_file}")
print(f"Patch Size: {PATCH_SIZE}x{PATCH_SIZE} (Radius: {PATCH_RADIUS})")
print(f"Target Train: {TRAIN_SAMPLES_PER_CLASS * 6:,} | Target Val: {VAL_SAMPLES_PER_CLASS * 6:,}")

# Verify paths
assert os.path.exists(labels_file), f"❌ Labels not found"
assert embeddings_dir.exists(), f"❌ Embeddings not found"

# Get and Split Tiles geographically
tile_files = sorted(list(embeddings_dir.glob("*.tif")))
np.random.seed(SEED)
np.random.shuffle(tile_files)

split_idx = int(len(tile_files) * TRAIN_SPLIT)
train_tiles = set(tile_files[:split_idx])
val_tiles = set(tile_files[split_idx:])

print(f"\n✓ Found {len(tile_files)} tiles")
print(f"  Training Tiles: {len(train_tiles)}")
print(f"  Validation Tiles: {len(val_tiles)}")
print("✅ Configuration validated!")

CONFIGURATION: CNN SPATIAL DATASET

Labels: /content/drive/MyDrive/bow_river_wetlands_10m_final.tif
Embeddings: /content/drive/MyDrive/EarthEngine
Output: /content/drive/MyDrive/wetland_cnn_dataset_15x15.npz
Patch Size: 15x15 (Radius: 7)
Target Train: 60,000 | Target Val: 12,000

✓ Found 100 tiles
  Training Tiles: 80
  Validation Tiles: 20
✅ Configuration validated!


In [ ]:
# CELL 3: Tile-Aware Coordinate Sampling
from rasterio.windows import Window

print("\n" + "="*70)
print("TILE-AWARE COORDINATE SAMPLING")
print("="*70)

train_coords = {c: {'y': [], 'x': []} for c in range(6)}
val_coords = {c: {'y': [], 'x': []} for c in range(6)}

def gather_coords(tile_set, coords_dict, desc):
    with rasterio.open(labels_file) as lbl_src:
        for tile_file in tqdm(tile_set, desc=desc):
            with rasterio.open(tile_file) as tile_src:
                if tile_src.count != 64: continue

                parts = tile_file.stem.split('-')
                if len(parts) < 3: continue

                r_off, c_off = int(parts[-2]), int(parts[-1])
                h, w = tile_src.height, tile_src.width

                # Read only the labels for this specific tile's footprint
                window = Window(c_off, r_off, w, h)
                tile_labels = lbl_src.read(1, window=window)

                r = PATCH_RADIUS
                for c in range(6):
                    y_local, x_local = np.where(tile_labels == c)

                    # Ensure coordinates aren't too close to the tile edge for patches
                    valid = (y_local >= r) & (y_local < h - r) & (x_local >= r) & (x_local < w - r)
                    if not valid.any(): continue

                    # Store as global coordinates
                    coords_dict[c]['y'].append(y_local[valid] + r_off)
                    coords_dict[c]['x'].append(x_local[valid] + c_off)

gather_coords(train_tiles, train_coords, "Scanning Train Tiles")
gather_coords(val_tiles, val_coords, "Scanning Val Tiles")

# Sample exactly the requested amounts per class
def balance_classes(coords_dict, target):
    sampled_y, sampled_x, sampled_l = [], [], []
    for c in range(6):
        if not coords_dict[c]['y']:
            print(f"⚠️ WARNING: No pixels found for class {c} in this split!")
            continue

        y_all = np.concatenate(coords_dict[c]['y'])
        x_all = np.concatenate(coords_dict[c]['x'])

        available = len(y_all)
        needed = min(target, available)

        if available < target:
            print(f"Notice: Class {c} only has {available:,} samples (Target: {target:,})")

        idx = np.random.choice(available, needed, replace=False)
        sampled_y.append(y_all[idx])
        sampled_x.append(x_all[idx])
        sampled_l.append(np.full(needed, c))

    return np.concatenate(sampled_y), np.concatenate(sampled_x), np.concatenate(sampled_l)

print("\nBalancing classes...")
train_y, train_x, train_labels = balance_classes(train_coords, TRAIN_SAMPLES_PER_CLASS)
val_y, val_x, val_labels = balance_classes(val_coords, VAL_SAMPLES_PER_CLASS)

print(f"\n✅ Final Coordinates - Train: {len(train_labels):,} | Val: {len(val_labels):,}")


TILE-AWARE COORDINATE SAMPLING


Scanning Train Tiles:  85%|████████▌ | 68/80 [01:50<00:10,  1.16it/s]

In [ ]:
# CELL 4: Extract PATCHES
import gc

print("\n" + "="*70)
print(f"EXTRACTING {PATCH_SIZE}x{PATCH_SIZE} PATCHES")
print("="*70)

max_train = len(train_labels)
max_val = len(val_labels)
temp_dir = "/content"
BATCH_SIZE = 1000

print(f"Allocating temp storage... (Train: {max_train:,}, Val: {max_val:,})")

X_train_mmap = np.lib.format.open_memmap(f'{temp_dir}/temp_X_train.npy', mode='w+', dtype='float32', shape=(max_train, 64, PATCH_SIZE, PATCH_SIZE))
y_train_mmap = np.lib.format.open_memmap(f'{temp_dir}/temp_y_train.npy', mode='w+', dtype='int64', shape=(max_train,))

X_val_mmap = np.lib.format.open_memmap(f'{temp_dir}/temp_X_val.npy', mode='w+', dtype='float32', shape=(max_val, 64, PATCH_SIZE, PATCH_SIZE))
y_val_mmap = np.lib.format.open_memmap(f'{temp_dir}/temp_y_val.npy', mode='w+', dtype='int64', shape=(max_val,))

train_ptr, val_ptr = 0, 0
all_tiles_sorted = sorted(list(train_tiles | val_tiles))

with tqdm(total=len(all_tiles_sorted), desc="Extracting", unit="tile") as pbar:
    for tile_file in all_tiles_sorted:
        is_train = tile_file in train_tiles

        # Select the correct coordinate pool based on the tile's split assignment
        y_inds, x_inds, lbls = (train_y, train_x, train_labels) if is_train else (val_y, val_x, val_labels)

        try:
            with rasterio.open(tile_file) as tile_src:
                if tile_src.count != 64:
                    pbar.update(1); continue

                r_off, c_off = int(tile_file.stem.split('-')[-2]), int(tile_file.stem.split('-')[-1])
                h, w = tile_src.height, tile_src.width
                r = PATCH_RADIUS

                in_tile_y = (y_inds >= r_off + r) & (y_inds < r_off + h - r)
                in_tile_x = (x_inds >= c_off + r) & (x_inds < c_off + w - r)
                mask = in_tile_y & in_tile_x

                if not mask.any():
                    pbar.update(1); continue

                tile_data = tile_src.read()

                valid_y = y_inds[mask] - r_off
                valid_x = x_inds[mask] - c_off
                valid_lbls = lbls[mask]

                for i in range(0, len(valid_y), BATCH_SIZE):
                    batch_y = valid_y[i : i + BATCH_SIZE]
                    batch_x = valid_x[i : i + BATCH_SIZE]
                    batch_labels = valid_lbls[i : i + BATCH_SIZE]

                    dy = np.arange(-r, r+1).reshape(1, 2*r+1, 1)
                    dx = np.arange(-r, r+1).reshape(1, 1, 2*r+1)

                    patch_y = batch_y[:, None, None] + dy
                    patch_x = batch_x[:, None, None] + dx

                    patches = tile_data[:, patch_y, patch_x].transpose(1, 0, 2, 3)
                    valid_patch_mask = ~np.isnan(patches).any(axis=(1,2,3))

                    if valid_patch_mask.any():
                        clean_patches = patches[valid_patch_mask]
                        clean_labels = batch_labels[valid_patch_mask]
                        n = len(clean_labels)

                        if is_train:
                            X_train_mmap[train_ptr : train_ptr+n] = clean_patches
                            y_train_mmap[train_ptr : train_ptr+n] = clean_labels
                            train_ptr += n
                        else:
                            X_val_mmap[val_ptr : val_ptr+n] = clean_patches
                            y_val_mmap[val_ptr : val_ptr+n] = clean_labels
                            val_ptr += n

                del tile_data
                gc.collect()

        except Exception as e:
            print(f"Error {tile_file.name}: {e}")

        pbar.update(1)

print(f"\n✅ Extraction Complete!")
print(f"Train samples saved: {train_ptr:,}")
print(f"Val samples saved:   {val_ptr:,}")

In [ ]:
# CELL 5: Save Final Compressed Dataset
print("\n" + "="*70)
print("SAVING FINAL DATASET")
print("="*70)

# Calculate weights using the data on disk
# We only read the labels into RAM, which is small
y_train_final = y_train_mmap[:train_ptr]
classes, counts = np.unique(y_train_final, return_counts=True)
weights = 1.0 / counts
weights = weights / weights.sum() * len(classes)
class_weights_dict = {k: v for k, v in zip(classes, weights)}

print("Class Weights (Train):")
print(class_weights_dict)

print(f"\nWriting to compressed file: {output_file}")
print("Streaming from disk... (This might take a few minutes)")

# SaveZ Compressed - numpy is smart enough to stream this from the memmaps
np.savez_compressed(
    output_file,
    X_train=X_train_mmap[:train_ptr],
    y_train=y_train_mmap[:train_ptr],
    X_val=X_val_mmap[:val_ptr],
    y_val=y_val_mmap[:val_ptr],
    class_weights=weights
)

# Cleanup temporary files to free up disk space
try:
    os.remove(f'{temp_dir}/temp_X_train.npy')
    os.remove(f'{temp_dir}/temp_y_train.npy')
    os.remove(f'{temp_dir}/temp_X_val.npy')
    os.remove(f'{temp_dir}/temp_y_val.npy')
    print("✓ Temporary files cleaned up")
except Exception as e:
    print(f"Warning cleaning up: {e}")

print(f"\n🎉 SAVED: {output_file}")
print(f"Ready for CNN training.")

In [ ]:
# CELL 6: Verify
print("\n" + "="*70)
print("VERIFICATION")
print("="*70)

data = np.load(output_file)

print(f"\nArrays: {list(data.keys())}")

for key in data.keys():
    arr = data[key]
    print(f"\n{key}:")
    print(f"  Shape: {arr.shape}")
    print(f"  Type: {arr.dtype}")

    if key == 'X':
        print(f"  Has NaN: {np.isnan(arr).any()} (should be False)")
        print(f"  Has Inf: {np.isinf(arr).any()} (should be False)")
        print(f"  Min: {arr.min():.4f}, Max: {arr.max():.4f}")
    elif key == 'y':
        print(f"  Classes: {np.unique(arr)}")

data.close()
print("\n✅ Verification passed!")